
# AI Task Manager — Add Tasks, Get AI Priorities, Deadlines and a Daily Plan

**Project | AI Mini Projects Series**

**Problem:** A to-do list does not tell you what to do first.

**What this notebook does:** You add tasks with deadlines and effort. It works out days left, gives a rule-based priority, then the AI gives its own priority with a reason, an order to follow, and one short suggestion. It can also plan your day for the hours you have.

**How it works:** tasks table (pandas) → days left + rule priority → LLM priority JSON → merge → sorted table → daily plan.

**Tools:** Python, pandas, Groq (openai/gpt-oss-120b), Google Colab

## Step 1: Install libraries

In [1]:
!pip install -q pandas openai

## Step 2: Enter Groq key

In [2]:
import os
import getpass
from openai import OpenAI

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
model_name = "openai/gpt-oss-120b"
print("Groq client ready")

Enter your Groq API key: ··········
Groq client ready


## Step 3: Helper functions

In [3]:
def ask_llm(system_msg, user_msg, temperature=0.3):
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}
            ],
            temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as e:
        return "Error calling Groq: " + str(e)

In [4]:
import json

def parse_json(text):
    text = text.replace("```json", "").replace("```", "").strip()
    list_pos = text.find("[")
    dict_pos = text.find("{")
    positions = []
    if list_pos != -1:
        positions.append(list_pos)
    if dict_pos != -1:
        positions.append(dict_pos)
    if len(positions) == 0:
        return None
    start = min(positions)
    if text[start] == "[":
        end = text.rfind("]")
    else:
        end = text.rfind("}")
    try:
        return json.loads(text[start:end + 1])
    except Exception as e:
        print("Could not read JSON:", e)
        return None

## Step 4: Load saved tasks or start with sample tasks
Sample deadlines are set from today's date, so the priorities always make sense.

In [5]:
import os
import datetime
import pandas as pd
from IPython.display import display

today = datetime.date.today()
print("Today:", today)

if os.path.exists("tasks.csv"):
    tasks = pd.read_csv("tasks.csv")
    tasks["deadline"] = pd.to_datetime(tasks["deadline"]).dt.date
    print("Loaded saved tasks:", len(tasks))
else:
    tasks = pd.DataFrame([
        [1, "Complete project report", today + datetime.timedelta(days=1), 4, "Pending"],
        [2, "Plan weekend trip", today + datetime.timedelta(days=5), 1, "Pending"],
        [3, "Learn AI tools", today + datetime.timedelta(days=10), 3, "Pending"],
        [4, "Workout", today, 1, "Pending"],
        [5, "Pay electricity bill", today + datetime.timedelta(days=2), 0.5, "Pending"],
        [6, "Prepare for interview", today + datetime.timedelta(days=3), 5, "Pending"]
    ], columns=["id", "task", "deadline", "hours", "status"])
    print("Started with sample tasks")

display(tasks)

Today: 2026-10-17
Started with sample tasks


,id,task,deadline,hours,status
0,1,Complete project report,2026-10-18,4.0,Pending
1,2,Plan weekend trip,2026-10-22,1.0,Pending
2,3,Learn AI tools,2026-10-27,3.0,Pending
3,4,Workout,2026-10-17,1.0,Pending
4,5,Pay electricity bill,2026-10-19,0.5,Pending
5,6,Prepare for interview,2026-10-20,5.0,Pending


## Step 5: Add your own tasks
Press Enter on the task name when you are done.

In [6]:
def add_task(name, deadline_text, hours):
    global tasks
    new_id = 1
    if len(tasks) > 0:
        new_id = int(tasks["id"].max()) + 1
    deadline = datetime.datetime.strptime(deadline_text, "%Y-%m-%d").date()
    new_row = pd.DataFrame([[new_id, name, deadline, float(hours), "Pending"]], columns=tasks.columns)
    tasks = pd.concat([tasks, new_row], ignore_index=True)
    print("Added task", new_id, ":", name)

while True:
    name = input("Task name (press Enter to stop adding): ").strip()
    if name == "":
        break
    d = input("Deadline (YYYY-MM-DD): ").strip()
    h = input("Hours needed (for example 2): ").strip()
    try:
        add_task(name, d, h)
    except Exception as e:
        print("Could not add. Use the date like 2026-10-25 and a number for hours.")

display(tasks)

Task name (press Enter to stop adding): 


,id,task,deadline,hours,status
0,1,Complete project report,2026-10-18,4.0,Pending
1,2,Plan weekend trip,2026-10-22,1.0,Pending
2,3,Learn AI tools,2026-10-27,3.0,Pending
3,4,Workout,2026-10-17,1.0,Pending
4,5,Pay electricity bill,2026-10-19,0.5,Pending
5,6,Prepare for interview,2026-10-20,5.0,Pending


## Step 6: Rule-based priority
Simple rule: due today or overdue = High, due within 3 days = Medium (High if it also needs 4+ hours), later = Low.

In [7]:
def rule_priority(days_left, hours):
    if days_left <= 0:
        return "High"
    if days_left <= 3:
        if hours >= 4 and days_left <= 2:
            return "High"
        return "Medium"
    return "Low"

pending = tasks[tasks["status"] == "Pending"].copy()
pending["days_left"] = pending["deadline"].apply(lambda d: (d - today).days)
pending["rule_priority"] = pending.apply(lambda row: rule_priority(row["days_left"], row["hours"]), axis=1)

display(pending[["id", "task", "deadline", "days_left", "hours", "rule_priority"]])

,id,task,deadline,days_left,hours,rule_priority
0,1,Complete project report,2026-10-18,1,4.0,High
1,2,Plan weekend trip,2026-10-22,5,1.0,Low
2,3,Learn AI tools,2026-10-27,10,3.0,Low
3,4,Workout,2026-10-17,0,1.0,High
4,5,Pay electricity bill,2026-10-19,2,0.5,Medium
5,6,Prepare for interview,2026-10-20,3,5.0,Medium


## Step 7: AI priority, order and suggestion

In [8]:
lines = []
for i, row in pending.iterrows():
    lines.append("ID " + str(row["id"]) + ": " + row["task"] + " | deadline " + str(row["deadline"]) + " | days left " + str(row["days_left"]) + " | needs " + str(row["hours"]) + " hours")
task_text = "\n".join(lines)

ai_prompt = (
    "Today is " + str(today) + ". Here are my pending tasks:\n" + task_text + "\n\n"
    "Decide the priority (High, Medium or Low) and the best order to do them. Think about deadlines, effort and what hurts most if missed. "
    "Return only JSON in this shape:\n"
    '{"suggestion": "one or two short sentences on what to focus on first and why", '
    '"tasks": [{"id": 1, "priority": "High", "order": 1, "reason": "short reason"}]}\n'
    "Include every task id exactly once."
)

ai_text = ask_llm("You are a practical task planning assistant. You reply with JSON only.", ai_prompt, 0.2)
ai_result = parse_json(ai_text)

pending["ai_priority"] = pending["rule_priority"]
pending["order"] = pending["days_left"].rank(method="first").astype(int)
pending["reason"] = "Based on deadline only (AI result not available)"
suggestion = "Start with the task that is due first."

if isinstance(ai_result, dict) and "tasks" in ai_result:
    suggestion = ai_result.get("suggestion", suggestion)
    for item in ai_result["tasks"]:
        try:
            mask = pending["id"] == int(item["id"])
            pending.loc[mask, "ai_priority"] = item.get("priority", "Medium")
            pending.loc[mask, "order"] = int(item.get("order", 99))
            pending.loc[mask, "reason"] = item.get("reason", "")
        except Exception as e:
            print("Skipped one AI item:", e)
else:
    print("Could not read the AI result, so the rule-based priority is used.")

pending = pending.sort_values("order")
display(pending[["order", "id", "task", "deadline", "ai_priority", "rule_priority", "reason"]])
print("AI SUGGESTION:", suggestion)

,order,id,task,deadline,ai_priority,rule_priority,reason
3,1,4,Workout,2026-10-17,High,High,Deadline is today; health routine should be do...
0,2,1,Complete project report,2026-10-18,High,High,"Due tomorrow; 4 h effort needed, missing it im..."
4,3,5,Pay electricity bill,2026-10-19,High,Medium,Due in two days; small time (0.5 h) but late p...
5,4,6,Prepare for interview,2026-10-20,High,Medium,Interview in three days; 5 h prep is critical ...
1,5,2,Plan weekend trip,2026-10-22,Medium,Low,Weekend trip planning due in five days; modera...
2,6,3,Learn AI tools,2026-10-27,Low,Low,Learning AI tools has the latest deadline and ...


AI SUGGESTION: Start with today's urgent tasks—Workout and the project report—then handle the bill and interview prep before moving to lower‑urgency items.


## Step 8: Plan my day

In [9]:
hours_text = input("How many hours do you have today? (press Enter for 4): ").strip()
try:
    free_hours = float(hours_text)
except:
    free_hours = 4

plan_prompt = (
    "I have " + str(free_hours) + " hours today. My pending tasks in priority order:\n" +
    "\n".join([str(r["order"]) + ". " + r["task"] + " (" + str(r["hours"]) + " h, due in " + str(r["days_left"]) + " days)" for i, r in pending.iterrows()]) +
    "\n\nMake a simple plan for today with time blocks. Do not plan more than " + str(free_hours) + " hours. "
    "Say clearly what to leave for tomorrow."
)

plan = ask_llm("You make realistic daily plans.", plan_prompt, 0.3)
print(plan)

How many hours do you have today? (press Enter for 4): 
**Today – 4 h total**

| Time (approx.) | Activity | Notes |
|----------------|----------|-------|
| 08:00 – 09:00  | **Workout** (1 h) | Stick to your usual routine – this is non‑negotiable. |
| 09:15 – 11:45  | **Project report – first chunk** (2.5 h) | Focus on the most critical sections (e.g., executive summary & key results). |
| 12:00 – 12:30  | **Pay electricity bill** (0.5 h) | Quick online payment; keep receipt. |
| 12:30 – 12:45  | **Buffer / transition** (0.15 h) | Short break, stretch, set up for tomorrow. |

*Total scheduled time: 4 h.*

---

### What to leave for tomorrow (or later)

- **Project report – remaining 1.5 h** (finish the analysis, conclusions, and formatting).  
- **Prepare for interview** (5 h total – start tomorrow).  
- **Plan weekend trip** (1 h).  
- **Learn AI tools** (3 h).  

You’ve covered the highest‑priority items that fit into today’s 4‑hour window, and the most urgent unfinished work (the re

## Step 9: Mark tasks as done and save
Type a task id to mark it done. Press Enter or type `exit` to stop.

In [10]:
while True:
    done_id = input("Task id to mark as done (press Enter or type exit to stop): ").strip()
    if done_id == "" or done_id.lower() == "exit":
        break
    try:
        tasks.loc[tasks["id"] == int(done_id), "status"] = "Done"
        print("Marked task", done_id, "as done")
    except Exception as e:
        print("Please type a number")

tasks.to_csv("tasks.csv", index=False)
print("Saved tasks.csv")
display(tasks)

try:
    from google.colab import files
    files.download("tasks.csv")
except Exception as e:
    print("Download not available here, file is saved in the notebook folder")

Task id to mark as done (press Enter or type exit to stop): 
Saved tasks.csv


,id,task,deadline,hours,status
0,1,Complete project report,2026-10-18,4.0,Pending
1,2,Plan weekend trip,2026-10-22,1.0,Pending
2,3,Learn AI tools,2026-10-27,3.0,Pending
3,4,Workout,2026-10-17,1.0,Pending
4,5,Pay electricity bill,2026-10-19,0.5,Pending
5,6,Prepare for interview,2026-10-20,5.0,Pending


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Key Points

- Days left and rule priority are calculated in code, so the basic priority never depends on the AI
- The AI adds a reason, an order and a suggestion. In my run the AI agreed with the rule on 3 of 6 tasks and raised the other 3 (the electricity bill and interview prep to High, the weekend trip to Medium). If the AI reply cannot be read, the notebook falls back to the rule-based priority (this did not happen in my run)
- The AI daily plan fitted the 4 hours I gave it, but it has small slips: it labelled a 15-minute buffer "0.15 h" and its suggestion called the project report a "today" task when it is due tomorrow. Read the plan before you follow it
- I only used the sample tasks. Adding tasks, marking tasks done and reloading a saved `tasks.csv` were not tested in my run
- Sample deadlines are set from today's date, so the demo always makes sense
- Colab clears files when the session ends, so download tasks.csv to keep your tasks

**Ideas to improve:** recurring tasks, calendar sync, reminders by email, and a Streamlit app with drag and drop.

---
Built by Akshat Kesharwani | GitHub: https://github.com/akshatkesharwani-info | Portfolio: https://akshatkesharwani-info.github.io/